# Feature-engine creation inside a pipeline

This notebook combines MathFeatures and RelativeFeatures with a gradient-boosting regressor. The pipeline applies the same formulas to training, test, and future data.

## Why use a pipeline?

A manually created ratio can be forgotten during inference or calculated with a different column order. Pipeline steps make the feature recipe reproducible and ensure that cross-validation refits the complete transformation on each training fold.

In [1]:
import importlib.util,subprocess,sys
required={'numpy':'numpy','pandas':'pandas','sklearn':'scikit-learn','feature_engine':'feature-engine'}
missing=[package for module,package in required.items() if importlib.util.find_spec(module) is None]
if missing: subprocess.check_call([sys.executable,'-m','pip','install',*missing])
from pathlib import Path
import numpy as np
import pandas as pd
from feature_engine.creation import MathFeatures,RelativeFeatures
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error,mean_squared_error,r2_score

paths=[Path('feature-engineering/imputation/houseprice.csv'),Path('../imputation/houseprice.csv'),Path('../../imputation/houseprice.csv'),Path('houseprice.csv')]
path=next((p for p in paths if p.exists()),None)
if path is None: raise FileNotFoundError('Place houseprice.csv in feature-engineering/imputation/.')
data=pd.read_csv(path,na_values=['NA'])
columns=['GrLivArea','LotArea','TotalBsmtSF','BsmtFinSF1','BsmtUnfSF','GarageArea','OverallQual']
model_data=data[columns+['SalePrice']].dropna().copy()
X=model_data[columns]; y=model_data['SalePrice']
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=.20,random_state=42)

## 1. Define the pipeline

The first MathFeatures step creates total and average area. The second creates a basement total. RelativeFeatures then creates land utilisation and finished-basement ratios. The final estimator receives the original and generated columns.

In [2]:
pipe=Pipeline([
    ('area_math',MathFeatures(variables=['GrLivArea','TotalBsmtSF'],func=['sum','mean'],new_variables_names=['total_living_basement_area','average_living_basement_area'],missing_values='ignore')),
    ('basement_math',MathFeatures(variables=['BsmtFinSF1','BsmtUnfSF'],func=['sum'],new_variables_names=['basement_components_total'],missing_values='ignore')),
    ('utilisation',RelativeFeatures(variables=['GrLivArea'],reference=['LotArea'],func=['div'],fill_value=0,missing_values='ignore')),
    ('finished_ratio',RelativeFeatures(variables=['BsmtFinSF1'],reference=['TotalBsmtSF'],func=['div'],fill_value=0,missing_values='ignore')),
    ('model',GradientBoostingRegressor(n_estimators=150,max_depth=2,learning_rate=.04,random_state=42))
])
pipe

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('area_math', ...), ('basement_math', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,variables,"['GrLivArea', 'TotalBsmtSF']"
,func,"['sum', 'mean']"
,new_variables_names,"['total_living_basement_area', 'average_living_basement_area']"
,missing_values,'ignore'
,drop_original,False
,variables,"['BsmtFinSF1', 'BsmtUnfSF']"
,func,['sum']


## 2. Fit and evaluate

Report the test metrics only after fitting. MAE is the average currency error, RMSE penalises large errors, and R² measures explained variation. Compare with a raw-feature baseline in a real project.

In [3]:
pipe.fit(X_train,y_train)
train_pred=pipe.predict(X_train); test_pred=pipe.predict(X_test)
metrics=pd.DataFrame({'Train':{'MAE':mean_absolute_error(y_train,train_pred),'RMSE':np.sqrt(mean_squared_error(y_train,train_pred)),'R2':r2_score(y_train,train_pred)},'Test':{'MAE':mean_absolute_error(y_test,test_pred),'RMSE':np.sqrt(mean_squared_error(y_test,test_pred)),'R2':r2_score(y_test,test_pred)}}).T
display(metrics.round(2))

,MAE,RMSE,R2
Train,18103.93,25476.54,0.89
Test,21642.82,34157.00,0.85


## 3. Inspect pipeline output

The pipeline can be sliced before the estimator to inspect the transformed dataframe and align model importance with column names.

In [4]:
transformed=pipe[:-1].transform(X_train)
new_columns=[c for c in transformed.columns if c not in X_train.columns]
print('Created columns:',new_columns)
display(transformed[new_columns].head())
importance=pd.Series(pipe.named_steps['model'].feature_importances_,index=transformed.columns).sort_values(ascending=False).head(12)
display(importance.to_frame('importance'))

Created columns: ['total_living_basement_area', 'average_living_basement_area', 'basement_components_total', 'GrLivArea_div_LotArea', 'BsmtFinSF1_div_TotalBsmtSF']


,total_living_basement_area,average_living_basement_area,basement_components_total,GrLivArea_div_LotArea,BsmtFinSF1_div_TotalBsmtSF
254,2628.0,1314.0,1314,0.156429,0.701674
1066,2370.0,1185.0,799,0.200459,0.000000
638,1592.0,796.0,796,0.090692,0.000000
799,2499.0,1249.5,731,0.245556,0.778386
380,2717.0,1358.5,1026,0.338200,0.212476


,importance
OverallQual,0.464196
total_living_basement_area,0.227343
average_living_basement_area,0.174792
BsmtFinSF1,0.039594
GarageArea,0.034959
GrLivArea,0.023466
LotArea,0.018325
GrLivArea_div_LotArea,0.011058
BsmtFinSF1_div_TotalBsmtSF,0.002338
TotalBsmtSF,0.001743


## Summary

Feature-engine transformers are compatible with scikit-learn Pipeline. This makes formula-based feature creation repeatable, inspectable, and safer during validation and deployment.